In [1]:
import torch
import math
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


Device: cuda


In [2]:
# AraGPT2 base — small, pure causal LM, the established Arabic perplexity tool
model_id = "aubmindlab/aragpt2-base"
ppl_tokenizer = AutoTokenizer.from_pretrained(model_id)
ppl_model = AutoModelForCausalLM.from_pretrained(model_id).to(device).eval()

print(f"Loaded {model_id}")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

C:\Users\aksha\anaconda3\envs\lingomatch\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\aksha\.cache\huggingface\hub\models--aubmindlab--aragpt2-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/553M [00:00<?, ?B/s]

Loaded aubmindlab/aragpt2-base
GPU memory: 0.55 GB


In [3]:
def compute_perplexity(text, model, tokenizer, device):
    """Compute perplexity of a text under the model.
    Low = natural Arabic; high = surprising/garbled."""
    if not text or not text.strip():
        return float('inf')
    
    encodings = tokenizer(text, return_tensors="pt").to(device)
    input_ids = encodings["input_ids"]
    
    # Need at least 2 tokens to compute a loss
    if input_ids.shape[1] < 2:
        return float('inf')
    
    with torch.inference_mode():
        outputs = model(input_ids, labels=input_ids)
        loss = outputs.loss  # average cross-entropy per token
    
    return math.exp(loss.item())

# Sanity check: clean Arabic should score LOW, gibberish should score HIGH
clean = "القانون هو مجموعة القواعد القانونية التي تنظم سلوك الأفراد في المجتمع."
gibberish = "كالمراه المقالت عالم الله المجادة عالم الا فالمتالات المالية المالية المالية"

print(f"Clean Arabic perplexity:     {compute_perplexity(clean, ppl_model, ppl_tokenizer, device):.1f}")
print(f"Gibberish perplexity:        {compute_perplexity(gibberish, ppl_model, ppl_tokenizer, device):.1f}")

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Clean Arabic perplexity:     25.2
Gibberish perplexity:        3333.6


In [4]:
# Each case: (name, OCR_output_text, surya_confidence, actual_CER)
# OCR strings + confidences from notebook 07; we now add perplexity as a 2nd signal.
cases = [
    ("Sample 43 (hallucinated repetition)",
     "كالمراه المقالت عالم الله المجادة عالم الا فالمتالات المالية المالية المالية المالية المالية المالية المالية المالية المالية المالية المالية المالية",
     0.3295, 2.8824),
    ("Sample 34 (recognition collapse)",
     "С",
     0.6235, 1.0000),
    ("Sample 12 (partial collapse)",
     "الع سه 88",
     0.4755, 0.7500),
    ("Sample 14 (Latin contamination)",
     "ws 10119 w 301 w111 تتحمل المسؤولية المتعلقة بخطة التحريري. وفضلا عن ذلك",
     0.1715, 0.6024),
    ("Sample 32 (diacritic scramble)",
     "وَدَالِءَ بِهَ دِينَةَ شَانَعُهَايَ سَلَمَ سَعَةَ التَّدْزِبِن لَقُرْصَ الرَّسْكِة",
     0.7651, 0.4306),
    ("Sample 13 (the uncatchable one)",
     "وَإِذا ما سَأَلَتْنِي عَن مَعْنَى لَفَظَهُ \"عرب\" عِنْدَ",  # clean-LOOKING Arabic, but wrong vs GT
     0.9972, 0.1184),
    ("Sample 2 (clean, good)",
     "تُثِير فِيها الاسود، وَهُوَ ما يَتَعارَض مَعَ اِكْتِشافِ",
     0.9953, 0.0000),
]

CONF_THRESHOLD = 0.92
PPL_THRESHOLD = 200   # provisional; we'll tune after seeing the numbers

print(f"{'Case':<40}{'CER':>7}{'Conf':>8}{'PPL':>9}  {'ConfFlag':>9}{'PplFlag':>9}")
print("-" * 90)

for name, text, conf, cer in cases:
    ppl = compute_perplexity(text, ppl_model, ppl_tokenizer, device)
    conf_flag = "FLAG" if conf < CONF_THRESHOLD else "pass"
    ppl_flag  = "FLAG" if ppl > PPL_THRESHOLD else "pass"
    ppl_str = f"{ppl:>8.0f}" if ppl != float('inf') else "     inf"
    print(f"{name:<40}{cer*100:>6.1f}%{conf:>8.2f}{ppl_str}  {conf_flag:>9}{ppl_flag:>9}")

Case                                        CER    Conf      PPL   ConfFlag  PplFlag
------------------------------------------------------------------------------------------
Sample 43 (hallucinated repetition)      288.2%    0.33     695       FLAG     FLAG
Sample 34 (recognition collapse)         100.0%    0.6211990433       FLAG     FLAG
Sample 12 (partial collapse)              75.0%    0.48 4603771       FLAG     FLAG
Sample 14 (Latin contamination)           60.2%    0.17     819       FLAG     FLAG
Sample 32 (diacritic scramble)            43.1%    0.77  452284       FLAG     FLAG
Sample 13 (the uncatchable one)           11.8%    1.00   83952       pass     FLAG
Sample 2 (clean, good)                     0.0%    1.00  126805       pass     FLAG


In [5]:
import re

def strip_diacritics(text):
    # Remove Arabic tashkeel (fatha, kasra, damma, sukun, shadda, tanwin, etc.)
    return re.sub(r'[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]', '', text)

# Re-score everything with diacritics stripped
print(f"{'Case':<40}{'CER':>7}{'Conf':>8}{'PPL(raw)':>11}{'PPL(stripped)':>15}")
print("-" * 90)
for name, text, conf, cer in cases:
    ppl_raw = compute_perplexity(text, ppl_model, ppl_tokenizer, device)
    ppl_strip = compute_perplexity(strip_diacritics(text), ppl_model, ppl_tokenizer, device)
    def fmt(p): return f"{p:>10.0f}" if p != float('inf') else "       inf"
    print(f"{name:<40}{cer*100:>6.1f}%{conf:>8.2f}{fmt(ppl_raw)}{fmt(ppl_strip):>15}")

Case                                        CER    Conf   PPL(raw)  PPL(stripped)
------------------------------------------------------------------------------------------
Sample 43 (hallucinated repetition)      288.2%    0.33       695            695
Sample 34 (recognition collapse)         100.0%    0.62  11990433       11990433
Sample 12 (partial collapse)              75.0%    0.48   4603771        4603771
Sample 14 (Latin contamination)           60.2%    0.17       819            819
Sample 32 (diacritic scramble)            43.1%    0.77    452284           5664
Sample 13 (the uncatchable one)           11.8%    1.00     83952            124
Sample 2 (clean, good)                     0.0%    1.00    126805            672
